# RAG(검색-증강 생성) Retrieval-Augmented Generation
- LLM이 외부 데이터를 컨텍스트로 활용.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"},
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [ ]:
%pip install -q "langchain_chroma>=0.1.2" langchain_community faiss-cpu

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# OpenAI 임베딩 모델
embedding = OpenAIEmbeddings(model='text-embedding-3-small')

# x = embedding.embed_query('점심 배불리 먹었더니 졸리다..')
# print(len(x), x)

# VectorStore 에 임베딩 후 저장(In memory)
vectorstore = Chroma.from_documents(
    documents,
    embedding=embedding
)

In [ ]:
vectorstore.similarity_search(
    '초보자가 키우기 좋은 애완동물 추천해줘', k=2
)

## 사전처리 단계
1. 문서 불러오기 (Loading)
2. 텍스트 나누기 (Splitting)
3. 숫자로 바꾸기 (Embedding)
4. 저장하기 (VectorStore)

In [ ]:
%pip install -q pymupdf

In [ ]:
# 1. Load
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader('./data/2025 취업트렌드.pdf')
docs = loader.load()

print('원본 pdf의 장수', len(docs))

원본 pdf의 장수 45


In [ ]:
# 2. Split
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 500글자당 1 청크 / 50글자는 겹치게 나눈다.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = text_splitter.split_documents(docs)

print('분할 후 청크 수', len(split_docs))

분할 후 청크 수 111


In [ ]:
# 3. 임베딩, 4. 벡터스토어 저장
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embedding = OpenAIEmbeddings()

vectorstore = FAISS.from_documents(documents=split_docs, embedding=embedding)

# Test
vectorstore.similarity_search(
    '데이터 분석가 2025년 채용 동향 예측 내용을 가져와봐', k=4
)

[Document(id='b3b48d6a-6b87-44f0-a7d7-63068174fc9d', metadata={'producer': 'Adobe PDF Library 18.0', 'creator': '', 'creationdate': '2024-12-31T04:48:13+00:00', 'source': './data/2025 취업트렌드.pdf', 'file_path': './data/2025 취업트렌드.pdf', 'total_pages': 45, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-12-31T04:48:14+00:00', 'trapped': '', 'modDate': 'D:20241231044814Z', 'creationDate': 'D:20241231044813Z', 'page': 1}, page_content='2025년도에는 문제 해결 능력, 유연한 사고와 협업 능력을 갖춘 인재가 주목받을 것이며, 맞춤형 채용 전략이 중요해질 것입니다.\n이 레포트가 다가오는 2025년도에 취준생들에게 채용 환경을 이해하고 미래를 준비하는 데 유용한 가이드가 되기를 기대합니다.\n제로베이스 드림.\n2024 취업 총 결산, 2025 채용 동향 예측 레포트'),
 Document(id='2dd055d7-66d9-4d1a-834f-15c6806a2365', metadata={'producer': 'Adobe PDF Library 18.0', 'creator': '', 'creationdate': '2024-12-31T04:48:13+00:00', 'source': './data/2025 취업트렌드.pdf', 'file_path': './data/2025 취업트렌드.pdf', 'total_pages': 45, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords':

## 검색 증강 단계
1. 사용자 질문 (Query)
2. 검색 (Retrieve)
3. LLM 
4. 최종 답변

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from langchain import hub

# Prompt 세팅 (랭체인 허브에서 가져오기)
prompt = hub.pull('rlm/rag-prompt')

# LLM 모델
llm = ChatOpenAI(model='gpt-4.1-nano')

# 검색기 생성(retriever 생성)
retriever = vectorstore.as_retriever()

chain = (
    {'context': retriever, 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

chain.invoke('올해 채용 트렌드에 대해 2줄로 요약해서 알려줘')

'올해 채용 트렌드에서는 경력직과 팀 협업 경험이 강조되고, 문제 해결 능력과 유연한 사고, 그리고 실무 중심의 평가 방식을 중요시합니다. 또한 맞춤형 채용 전략과 최신 기술 트렌드를 접목한 프로젝트 참여가 경쟁력을 높이는 핵심 요소로 나타나고 있습니다.'

In [ ]:
# Agent + RAG
from langchain.agents import create_openai_tools_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.memory import ConversationBufferMemory
from langchain_tavily import TavilySearch
from datetime import datetime
# RAG 관련
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.tools.retriever import create_retriever_tool

today = datetime.today().strftime('%Y-%m-%d')

llm = ChatOpenAI(model='gpt-4.1-nano')

search_tool = TavilySearch(
    max_results=5,
    topic='general'
)

loader = PyMuPDFLoader('./data/2025 취업트렌드.pdf')
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = splitter.split_documents(docs)
embedding = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(split_docs, embedding=embedding)
retriever = vectorstore.as_retriever()

rag_tool = create_retriever_tool(
    retriever,
    name='pdf_search',
    description='PDF 문서에서 질문과 관련된 내용을 검색합니다.'  # Agent가 언제 이 tool을 쓸지 알게됨
)

text = f"""
너는 웹 검색이 가능하고, 2025년 최신 채용 정보를 담은 pdf 를 검색할 수 있는 어시스턴트야.

- 사용자가 PDF 문서와 관련된 질문(ex. '이 pdf에서', '문서내용', '파일에서)을 하면 반드시 'pdf_search' 도구를 써야해
- 사용자 질문이 팩트체크를 필요로 하고, 최신성이 필요하다 판단되면 web_search를 실행해야해
- 사용자가 일반적인 질문을 하고, 최신성이나 팩트체크가 필요없으면 그냥 답변해
- 뭔가 확실하지 않으면 pdf_search 와 web_search를 모두 실행해서 답변을 생성해
- 간단, 요약해서 답변을 생성해

오늘은 {today} 야.
"""

prompt = ChatPromptTemplate.from_messages([
    ('system', text),
    MessagesPlaceholder(variable_name='chat_history'),
    ('human', '{input}'),
    MessagesPlaceholder(variable_name='agent_scratchpad')  # 도구(검색) 호출때 필요함
])

memory = ConversationBufferMemory(
    return_messages=True,
    memory_key='chat_history'
)

agent = create_openai_tools_agent(
    llm=llm,
    tools=[search_tool, rag_tool],
    prompt=prompt,
)

agent_executor = AgentExecutor(
    agent=agent,
    memory=memory, 
    tools=[search_tool, rag_tool],
    verbose=True)

In [ ]:
agent_executor.invoke({'input': '기획자가 데이터 분석가로 취업한다면, 기획자의 어떤 경험이 데이터 분석가에 도움이 될까?'})



> Entering new AgentExecutor chain...
기획자의 경험 중에서 데이터 분석과 관련된 부분은 문제 해결 능력, 데이터 기반 의사결정 능력, 프로젝트 관리 경험, 그리고 사용자 니즈 파악 능력 등입니다.

> Finished chain.


{'input': '기획자가 데이터 분석가로 취업한다면, 기획자의 어떤 경험이 데이터 분석가에 도움이 될까?',
 'chat_history': [HumanMessage(content='파일에서 데이터분석 분야 현직자 조언이 뭐래?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='현직 데이터 분석가들은 조언으로 다음과 같이 말하고 있습니다.  \n\n- 이직을 고려할 때는 목표 회사와 직무를 신중히 설정하는 것이 중요하며, 무리한 이직은 실패로 이어질 수 있음. 현재에 만족하거나 작은 변화만 원한다면 내부에서 직무 변경이나 업무 확장을 고려하라고 조언.  \n- 지금 하는 일이 왜 중요한지 스스로 답을 찾고, 조급해하지 말고 경험을 자산으로 삼으며 자기 계발과 업계 동향 파악이 필요. 시장 상황에 따라 채용 기회가 달라지니 트렌드에 민감해야 함.  \n- AI 시대에서도 분석역량과 멀티플레이어적 능력에 대한 수요는 계속 늘어날 것이며, 긍정적인 인상을 주는 적극적 태도와 문제 해결 역량이 중요.  \n\n현직자들은 어려운 시기일수록 자신감과 준비된 태도를 유지하며, 유의미한 임팩트를 낼 수 있는 역할을 잘 준비하는 것이 핵심이라고 강조합니다.', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='파일에서 데이터분석 분야 현직자 조언이 뭐래? 한 줄로 요약해', additional_kwargs={}, response_metadata={}),
  AIMessage(content='데이터 분석 분야 현직자들은 "경험을 자산으로 삼고, 시장 트렌드와 자기 계발에 민감하며, 문제 해결 역량과 긍정적 인상으로 임팩트를 만들어야 한다"고 조언합니다.', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='파일에서 데이터분석 분야 현직자 조언이 뭐래? 한 줄만 